# 이진 분류 실험 ⑧ — 피처 묶음 비교 (feature groups)
v10(태훈님) 과 내 실험에 **공통으로 있는 피처를 기준선**으로 두고, 한쪽에만 있는 피처를 **묶음 단위로 하나씩 얹어** 효과를 본다. 트리(LightGBM) 만, 시드 3개.

| 조건 | 피처 | 열 수 |
|---|---|---:|
| `base` | 공통 17종 (v10 이름 기준) | 23 |
| `base+A` | + 단기 상대 수 · 24시간 건수 (내 실험) | 26 |
| `base+B` | + 시간대 (내 실험) | 26 |
| `base+C` | + 소유주 (v10) | 29 |
| `base+D` | + 유형 · 국가 (v10) | 37 |
| `base+E` | + 당일 흐름 · 금액 대비 (v10 신규) | 28 |
| `base+F` | + 반복 금액 (v10 신규) | 26 |
| `base+G` | + 나머지 v10 피처 | 36 |

- **결제방식(`payment_format`) 과 절대 시각(`timestamp`) 은 어느 조건에도 넣지 않는다.** 결제방식이 있으면 다른 피처의 효과가 가려진다 (07 결과)
- 데이터는 무패턴 분석 06(`code/nonpaten_vs_normal/06_lgbm_nonpattern.ipynb`) 이 만든 표(v10 41종 · 64열) 에 내 피처 6열을 `edge_id` 로 붙여 쓴다
- 분할 · 검증 표본 · 테스트 구간은 지금까지와 같다 (Train 8/8~10/2 · Val 10/3~10/16 · Test 10/17~10/30 전체)
- 학습은 양성 전부 + 정상 10% 표본(가중치 10). 06 과 같은 표본이라 조건 · 시드가 달라도 학습 행은 같다
- **주의**: v10 노드 피처는 구간 끝 기준 프로필이라 구간 안의 미래 활동이 섞여 있다. 모든 조건에 똑같이 들어가므로 조건 간 비교는 되지만 절대 수치는 낙관적이다
- 1단계(묶음을 하나씩 얹기) → 살아남은 묶음을 `KEEP` 에 적고 → 2단계(다 합친 뒤 하나씩 빼기)

## 0. 작업 준비

In [ ]:
from pathlib import Path
import json, time, gc
import duckdb, numpy as np, pandas as pd, pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import average_precision_score
from IPython.display import display

In [ ]:
# ---- 실험 설정 ------------------------------------------------------------
DATASET = 'HI-Large'
PROJECT_ROOT = None            # 자동 탐색 실패 시 Path('/home/donghyeon/workspace/money_laundry_ai_')
SMOKE = False                  # True: 학습 20만 행 · 30 라운드 · 테스트 하루치 · 시드 1개로 끝까지 한 번 (에러 잡기용)
SEEDS = [0] if SMOKE else [0, 1, 2]
THREADS, MEMORY = 8, '10GB'    # LightGBM 스레드 / DuckDB 메모리
BUDGETS = (0.001, 0.01)        # 알림 예산 = 테스트 전체 상위 비율. 0.1% ≈ 2.7만 건(양성 수 규모), 1% = 논문 관행

# 무패턴 분석 06 과 같은 설정 (num_leaves 63, max_delta_step 0.7, scale_pos_weight 6). 시드는 실행마다 바꿔 넣는다
LGB_PARAMS = dict(objective='binary', learning_rate=0.05, num_leaves=63, min_child_weight=1.0, reg_lambda=1.0,
                  bagging_fraction=0.8, bagging_freq=1, feature_fraction=1.0, max_delta_step=0.7,
                  scale_pos_weight=6, metric='average_precision', num_threads=THREADS, verbose=-1)
ROUNDS, EARLY_STOP = (30, 10) if SMOKE else (1000, 50)

In [ ]:
# ---- 피처 묶음 ------------------------------------------------------------
both = lambda names: [f'src_{n}' for n in names] + [f'dst_{n}' for n in names]      # 계좌 피처는 송신 · 수신 양쪽 열

# 공통 17종 (v10 이름). 괄호는 내 실험의 대응 피처
BASE = (['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count',          # 같은 쌍 이력 4 (02 와 동일)
         'sender_count_1h', 'receiver_count_1h',                                                    # 1시간 건수 (src_out_1h, dst_in_1h)
         'sender_gap_minutes', 'receiver_gap_minutes',                                              # 직전 간격 (src/dst_gap_minutes)
         'paid_currency_z',                                                                         # 거래 금액 (amount_received)
         'minute_sender_vs_prior_hour', 'minute_receiver_vs_prior_hour',                            # 같은 분 집중도 (src/dst_minute_burst)
         'src_day_ccy_rate_vs_prior', 'dst_day_ccy_rate_vs_prior']                                  # 평소 대비 활동량 (src/dst_rate_vs_prior)
        + both(['log_out_count', 'log_in_count', 'log_out_neighbors', 'log_in_neighbors', 'log_active_days']))   # 누적 건수 · 상대 수 · 활동일 (hist_*)

GROUPS = {
    'A': ['src_uniq_dst_1d', 'dst_uniq_src_1d', 'src_out_1d'],                                       # 내 실험 03: 24시간 상대 계좌 수 · 송금 건수
    'B': ['hour_sin', 'hour_cos', 'weekday'],                                                        # 내 실험 07: 시간대
    'C': both(['log_entity_observed_accounts', 'log_entity_out_diversity', 'log_entity_in_diversity']),   # v10 소유주 (KYC)
    'D': both(['type_Corporation', 'type_Partnership', 'type_Sole_Proprietorship', 'type_Individual', 'type_Country', 'type_Direct'])
         + ['same_known_country', 'both_countries_known'],                                           # v10 유형 · 국가 (KYC)
    'E': ['src_day_ccy_flow_balance', 'dst_day_ccy_flow_balance', 'src_day_ccy_log_matching_in_amount',
          'src_prior_ccy_amount_vs_mean', 'dst_prior_ccy_amount_vs_mean'],                           # v10 신규: 당일 흐름 · 금액 대비
    'F': ['src_prior_ccy_amount_seen', 'dst_prior_ccy_amount_seen', 'pair_amount_seen_before_day'],  # v10 신규: 반복 금액
    'G': ['pair_age_at_day_start', 'same_currency', 'is_self_transfer']
         + both(['log_self_count', 'log_reciprocal_neighbors', 'mean_paid_currency_z', 'paid_outlier_fraction', 'log_currency_count']),   # 나머지 v10
}
GROUP_NAMES = {'A': '단기 상대 수·24h 건수 (나)', 'B': '시간대 (나)', 'C': '소유주 (v10)', 'D': '유형·국가 (v10)',
               'E': '당일 흐름·금액 대비 (v10)', 'F': '반복 금액 (v10)', 'G': '나머지 v10'}
MINE_ACT, MINE_TOD = GROUPS['A'], GROUPS['B']                 # features_bc 에서 붙일 내 피처
ALL_FEATS = BASE + [c for g in GROUPS.values() for c in g]
V10_FEATS = [c for c in ALL_FEATS if c not in MINE_ACT + MINE_TOD]
assert len(set(ALL_FEATS)) == len(ALL_FEATS) == 70 and len(V10_FEATS) == 64 and len(BASE) == 23
META = ['edge_id', 'y', 'cls', 'hot', 'fmt', 'w']             # cls 0 정상 · 1 패턴 · 2 무패턴, fmt 결제방식(평가에서만 씀), w 표본 가중치
{k: len(v) for k, v in GROUPS.items()}

In [ ]:
# ---- 경로 ----------------------------------------------------------------
def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / 'parquets' / DATASET / 'transactions.parquet').is_file():
            return c
    raise FileNotFoundError('transactions.parquet 를 찾지 못했다. PROJECT_ROOT 를 지정')


ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else find_root()
BASE_DIR = ROOT / 'parquets' / DATASET
SRC06 = BASE_DIR / 'nonpaten_vs_normal' / '06_lgbm'           # 무패턴 분석 06 의 데이터셋 (train.parquet, val.parquet, test/<날짜>.parquet)
FBC = BASE_DIR / 'features_bc'                                # binary_classification 단계별 피처
ACT_PATH, TOD_PATH = FBC / 'account_activity.parquet', FBC / 'time_of_day.parquet'
tag = 'feature_groups' + ('_smoke' if SMOKE else '')
DATA = BASE_DIR / tag                                         # 내 피처를 붙인 표 + 모델 · 점수 (용량이 커서 git 밖)
RUNS = DATA / 'runs'
OUT_DIR = Path.cwd() / 'results' / DATASET / tag              # 요약 표 (git 에 올리는 작은 파일)
for p in (DATA / 'test', RUNS, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT); print('DATA:', DATA); print('OUT :', OUT_DIR)

## 1. 입력 확인
필요한 것 — 없으면 아래 순서로 먼저 만든다

| 파일 | 만드는 곳 |
|---|---|
| `parquets/HI-Large/features_{train,val,test}/` | `code/02_build_features.ipynb` (분할별) — 받아 둔 `features_split.tar` 를 풀어도 됨 |
| `nonpaten_vs_normal/05_features/` | `code/nonpaten_vs_normal/05_nonpattern_features.ipynb` 전체 |
| `nonpaten_vs_normal/06_lgbm/train.parquet` · `val.parquet` · `test/` | `06_lgbm_nonpattern.ipynb` 의 **3. 데이터셋** 절까지 (학습 셀은 안 돌려도 됨) |
| `features_bc/account_activity.parquet` | `03_account_activity.ipynb` 피처 계산 셀 |
| `features_bc/time_of_day.parquet` | `07_tree_ablations.ipynb` 1절 |

In [ ]:
test_src = sorted((SRC06 / 'test').glob('*.parquet'))
need = {'06 train': SRC06 / 'train.parquet', '06 val': SRC06 / 'val.parquet', 'account_activity': ACT_PATH, 'time_of_day': TOD_PATH}
missing = [k for k, p in need.items() if not p.exists()] + ([] if test_src else ['06 test/'])
assert not missing, f'입력이 없다: {missing}'
if SMOKE:
    test_src = test_src[:1]
# 열 이름 확인 — 06 표에 v10 64열 + META, features_bc 에 내 6열
cols06 = set(pq.ParquetFile(SRC06 / 'train.parquet').schema_arrow.names)
assert not (set(V10_FEATS + META) - cols06), f'06 표에 없는 열: {sorted(set(V10_FEATS + META) - cols06)}'
assert not (set(MINE_ACT) - set(pq.ParquetFile(ACT_PATH).schema_arrow.names)), 'account_activity 열 확인'
assert not (set(MINE_TOD) - set(pq.ParquetFile(TOD_PATH).schema_arrow.names)), 'time_of_day 열 확인'
pd.Series({'06 train 행': pq.ParquetFile(SRC06 / 'train.parquet').metadata.num_rows, '06 val 행': pq.ParquetFile(SRC06 / 'val.parquet').metadata.num_rows,
           '06 test 파일': len(test_src), '06 test 행': sum(pq.ParquetFile(f).metadata.num_rows for f in test_src)}).map('{:,}'.format)

## 2. 내 피처 붙이기 — `edge_id` 조인
06 표의 각 행에 `account_activity`(A 묶음 3열) 와 `time_of_day`(B 묶음 3열) 를 붙여 `parquets/HI-Large/feature_groups/` 에 저장한다.
`features_bc` 파일은 1.8억 행이라 그 표의 `edge_id` 범위만 읽는다 (`edge_id` 는 시각순이라 날짜 하나가 연속 구간). 이미 만든 파일은 건너뛴다

In [ ]:
con = duckdb.connect()
con.execute(f"SET threads={THREADS}"); con.execute(f"SET memory_limit='{MEMORY}'"); con.execute(f"SET temp_directory='{DATA / 'tmp'}'")
q = lambda v: "'" + str(v).replace("'", "''") + "'"


def attach(src, dst):
    # src(06 표) 에 내 피처 6열을 붙여 dst 로. 행 수가 줄면 features_bc 에 그 edge_id 가 없다는 뜻이므로 멈춘다
    n_src = pq.ParquetFile(src).metadata.num_rows
    if dst.exists():
        assert pq.ParquetFile(dst).metadata.num_rows == n_src, f'{dst.name} 행 수가 다르다 — 지우고 다시'
        return n_src
    lo, hi = con.execute(f"SELECT min(edge_id), max(edge_id) FROM read_parquet({q(src)})").fetchone()
    part = dst.with_suffix('.partial')
    con.execute(f'''
        COPY (
            SELECT d.*, {', '.join('a.' + c for c in MINE_ACT)}, {', '.join('t.' + c for c in MINE_TOD)}
            FROM read_parquet({q(src)}) d
            JOIN (SELECT edge_id, {', '.join(MINE_ACT)} FROM read_parquet({q(ACT_PATH)}) WHERE edge_id BETWEEN {lo} AND {hi}) a USING (edge_id)
            JOIN (SELECT edge_id, {', '.join(MINE_TOD)} FROM read_parquet({q(TOD_PATH)}) WHERE edge_id BETWEEN {lo} AND {hi}) t USING (edge_id)
            ORDER BY d.edge_id
        ) TO {q(part)} (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)''')
    part.rename(dst)
    n = pq.ParquetFile(dst).metadata.num_rows
    assert n == n_src, f'{dst.name}: {n:,} / {n_src:,} 행 — features_bc 에 없는 edge_id 가 있다'
    return n

In [ ]:
t0 = time.time()
print(f"train {attach(SRC06 / 'train.parquet', DATA / 'train.parquet'):,} 행 ({time.time() - t0:.0f}s)", flush=True)
print(f"val   {attach(SRC06 / 'val.parquet', DATA / 'val.parquet'):,} 행 ({time.time() - t0:.0f}s)", flush=True)
n_test = 0
for f in test_src:
    n_test += attach(f, DATA / 'test' / f.name)
print(f'test  {n_test:,} 행 ({time.time() - t0:.0f}s)')
con.close()
test_files = [DATA / 'test' / f.name for f in test_src]

## 3. 학습 · 검증 표 적재

In [ ]:
def load(path, cols):
    return pq.read_table(path, columns=cols).to_pandas(strings_to_categorical=True)      # fmt 문자열 → category (메모리)


tr = load(DATA / 'train.parquet', META + ALL_FEATS)
va = load(DATA / 'val.parquet', META + ALL_FEATS)
if SMOKE:
    tr = pd.concat([tr[tr.y == 1], tr[tr.y == 0].sample(200_000, random_state=0)]).reset_index(drop=True) if (tr.y == 0).sum() > 200_000 else tr
for nm, d in [('train', tr), ('val', va)]:
    print(f"{nm}: {len(d):,} 행 · 양성 {int(d.y.sum()):,} (패턴 {int((d.cls == 1).sum()):,}, 무패턴 {int((d.cls == 2).sum()):,}) · 가중 정상 {d.w[d.y == 0].sum():,.0f}")
bad = {c: int((~np.isfinite(tr[c].to_numpy(np.float64))).sum()) for c in ALL_FEATS}
assert sum(bad.values()) == 0, {k: v for k, v in bad.items() if v}

In [ ]:
# 붙인 피처가 의도대로 들어왔나 — 라벨별 평균 (정상 / 패턴 / 무패턴). A 묶음은 패턴 쪽이 커야 한다 (03 결과)
tr.groupby('cls')[MINE_ACT + MINE_TOD].mean().T.rename(columns={0: '정상', 1: '패턴', 2: '무패턴'}).round(3)

## 4. 실행 함수
- `train_runs(conds)`: 조건 × 시드마다 LightGBM 학습 → 모델(`.txt`) · 검증 점수 · 정보(`.json`) 저장. **끝난 실행은 건너뛴다** (커널이 죽어도 다시 실행하면 이어짐)
- `score_runs(conds)`: 테스트 날짜 파일을 **한 번만** 읽으면서 아직 점수가 없는 모델 전부의 점수를 낸다
- `eval_runs(conds)`: 점수 → 지표 표 (실행 하나당 한 행)
- `summarize(runs, base)`: 시드 평균 ± 표준편차, 기준 조건 대비 차이와 판정

In [ ]:
run_name = lambda cond, seed: f'{cond}_s{seed}'


def train_runs(conds):
    for cond, feats in conds.items():
        for seed in SEEDS:
            name = run_name(cond, seed)
            if (RUNS / f'{name}.json').exists():
                print(f'[{name}] 이미 완료 — 건너뜀'); continue
            t0 = time.time()
            params = dict(LGB_PARAMS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed, data_random_seed=seed)
            dtr = lgb.Dataset(tr[feats].to_numpy(np.float32), label=tr.y.to_numpy(), weight=tr.w.to_numpy(), feature_name=feats, free_raw_data=True)
            dva = lgb.Dataset(va[feats].to_numpy(np.float32), label=va.y.to_numpy(), weight=va.w.to_numpy(), reference=dtr)
            ev = {}
            m = lgb.train(params, dtr, num_boost_round=ROUNDS, valid_sets=[dva], valid_names=['val'],
                          callbacks=[lgb.early_stopping(EARLY_STOP, first_metric_only=True, verbose=False), lgb.record_evaluation(ev)])
            m.save_model(str(RUNS / f'{name}.txt'), num_iteration=m.best_iteration)
            np.save(RUNS / f'{name}_val.npy', m.predict(va[feats].to_numpy(np.float32), num_iteration=m.best_iteration).astype(np.float32))
            imp = dict(zip(feats, m.feature_importance('gain', iteration=m.best_iteration).tolist()))
            info = dict(cond=cond, seed=seed, n_feat=len(feats), best_iteration=int(m.best_iteration), val_AP=float(max(ev['val']['average_precision'])),
                        train_sec=round(time.time() - t0), features=feats, gain=imp)
            (RUNS / f'{name}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')      # 마지막에 쓰는 파일 — 있으면 끝까지 간 실행
            print(f"[{name}] {len(feats)}열  best_it {m.best_iteration:4d}  val AP {info['val_AP']:.4f}  ({info['train_sec']}s)", flush=True)
            del dtr, dva, m; gc.collect()

In [ ]:
TEST_META = RUNS / 'test_meta.parquet'            # 테스트 행 순서의 edge_id · y · cls · hot · fmt (점수 .npy 와 같은 순서)


def score_runs(conds):
    names = [run_name(c, s) for c in conds for s in SEEDS]
    pending = [n for n in names if not (RUNS / f'{n}_test.npy').exists()]
    if not pending and TEST_META.exists():
        print('테스트 점수 모두 있음'); return
    infos = {n: json.loads((RUNS / f'{n}.json').read_text(encoding='utf-8')) for n in pending}
    boosters = {n: lgb.Booster(model_file=str(RUNS / f'{n}.txt')) for n in pending}
    need = sorted({c for n in pending for c in infos[n]['features']})
    parts, meta_parts, t0 = {n: [] for n in pending}, [], time.time()
    for i, f in enumerate(test_files):
        d = load(f, META + need)
        meta_parts.append(d[['edge_id', 'y', 'cls', 'hot', 'fmt']])
        for n in pending:
            parts[n].append(boosters[n].predict(d[infos[n]['features']].to_numpy(np.float32)).astype(np.float32))
        del d; gc.collect()
        print(f'  {f.stem}  ({i + 1}/{len(test_files)}, {time.time() - t0:.0f}s)', flush=True)
    meta = pd.concat(meta_parts, ignore_index=True)
    if TEST_META.exists():
        assert np.array_equal(pd.read_parquet(TEST_META, columns=['edge_id'])['edge_id'].to_numpy(), meta['edge_id'].to_numpy()), '테스트 행 순서가 이전 실행과 다르다'
    else:
        meta.to_parquet(TEST_META, index=False)
    for n in pending:
        np.save(RUNS / f'{n}_test.npy', np.concatenate(parts[n]))
    print(f'테스트 점수 {len(pending)}개 저장 · {len(meta):,} 행 ({time.time() - t0:.0f}s)')

In [ ]:
def evaluate(score, y, cls, is_ach):
    # 테스트 점수 하나 → 지표. 패턴 기준 지표는 무패턴 양성을 정답도 오답도 아닌 것으로 두고 알림에서도 뺀다
    out = {'test_AP': average_precision_score(y, score)}
    m = cls != 2; out['AP_패턴'] = average_precision_score(cls[m] == 1, score[m])            # 정상 vs 패턴 (무패턴 제외)
    m = cls != 1; out['AP_무패턴'] = average_precision_score(cls[m] == 2, score[m])          # 정상 vs 무패턴 (패턴 제외)
    n_pat, n_un = int((cls == 1).sum()), int((cls == 2).sum())
    for b in BUDGETS:
        k = f'{b * 100:g}%'
        hit = score >= np.quantile(score, 1 - b)
        tp_pat, tp_un, fp = int((hit & (cls == 1)).sum()), int((hit & (cls == 2)).sum()), int((hit & (cls == 0)).sum())
        out[f'알림@{k}'] = int(hit.sum())                                                   # 동점이 많으면 예산보다 커진다
        out[f'패턴R@{k}'] = tp_pat / max(n_pat, 1)
        out[f'패턴P@{k}'] = tp_pat / max(tp_pat + fp, 1)
        out[f'정상오탐@{k}'] = fp
        out[f'무패턴R@{k}'] = tp_un / max(n_un, 1)
        out[f'비ACH알림@{k}'] = int((hit & ~is_ach).sum())
        out[f'비ACH양성R@{k}'] = int((hit & (y == 1) & ~is_ach).sum()) / max(int(((y == 1) & ~is_ach).sum()), 1)
    return out


def eval_runs(conds):
    te = pd.read_parquet(TEST_META)
    y, cls, is_ach = te['y'].to_numpy().astype(np.int8), te['cls'].to_numpy().astype(np.int8), (te['fmt'].astype(str) == 'ACH').to_numpy()
    rows = []
    for cond in conds:
        for seed in SEEDS:
            name = run_name(cond, seed)
            info = json.loads((RUNS / f'{name}.json').read_text(encoding='utf-8'))
            s = np.load(RUNS / f'{name}_test.npy'); assert len(s) == len(te)
            rows.append(dict(cond=cond, seed=seed, n_feat=info['n_feat'], best_iteration=info['best_iteration'], val_AP=info['val_AP'], **evaluate(s, y, cls, is_ach)))
    return pd.DataFrame(rows)

In [ ]:
KEY = ['val_AP', 'test_AP', 'AP_패턴', 'AP_무패턴', '패턴R@0.1%', '패턴P@0.1%', '무패턴R@0.1%', '무패턴R@1%', '비ACH알림@0.1%']   # 요약에 보여 줄 지표
JUDGE = ['test_AP', 'AP_패턴', '패턴P@0.1%', '무패턴R@1%']                                                                       # 판정에 쓰는 지표


def summarize(runs, base):
    # 시드 평균 · 표준편차, 기준 조건(base) 대비 차이. 판정: 차이가 2 × max(두 조건의 시드 표준편차) 를 넘으면 ↑ / ↓, 아니면 ·
    g = runs.groupby('cond')[KEY]
    mean, std = g.mean(), g.std(ddof=1).fillna(0.0)
    delta = mean[JUDGE] - mean.loc[base, JUDGE]
    noise = 2 * std[JUDGE].clip(lower=std.loc[base, JUDGE], axis=1)          # 열마다 두 조건 중 큰 표준편차
    verdict = pd.DataFrame(np.where(delta > noise, '↑', np.where(delta < -noise, '↓', '·')), index=delta.index, columns=delta.columns)
    verdict.loc[base] = ''
    order = list(dict.fromkeys(runs['cond']))
    return mean.loc[order], std.loc[order], delta.loc[order], verdict.loc[order]

## 5. 1단계 — 공통 기준선에 묶음을 하나씩 얹기
8개 조건 × 시드 3개 = 24번 학습. 무패턴 분석 06 기준 모델 하나에 7~8분(52~70열) 이었으니 3시간 안팎 + 테스트 점수 계산

In [ ]:
STAGE1 = {'base': BASE, **{f'base+{k}': BASE + g for k, g in GROUPS.items()}}
pd.DataFrame({'열 수': {k: len(v) for k, v in STAGE1.items()}, '묶음': {'base': '공통 17종', **{f'base+{k}': v for k, v in GROUP_NAMES.items()}}})

In [ ]:
train_runs(STAGE1)

In [ ]:
score_runs(STAGE1)

In [ ]:
runs1 = eval_runs(STAGE1)
runs1.to_csv(OUT_DIR / 'stage1_runs.csv', index=False, encoding='utf-8-sig')
runs1.round(4)

In [ ]:
# 시드 평균 (괄호 없이) 과 표준편차
mean1, std1, delta1, verdict1 = summarize(runs1, 'base')
mean1.to_csv(OUT_DIR / 'stage1_mean.csv', encoding='utf-8-sig'); std1.to_csv(OUT_DIR / 'stage1_std.csv', encoding='utf-8-sig')
display(mean1.round(4)); display(std1.round(4))

In [ ]:
# 기준선 대비 차이와 판정 (↑ 오름 · ↓ 내림 · · 시드 편차 안)
judge1 = pd.concat({'차이': delta1.round(4), '판정': verdict1}, axis=1)
judge1.to_csv(OUT_DIR / 'stage1_judge.csv', encoding='utf-8-sig')
judge1

In [ ]:
# 묶음 안에서 어떤 피처를 썼나 — 조건별 gain 상위 8 (시드 0)
for cond in STAGE1:
    info = json.loads((RUNS / f'{run_name(cond, SEEDS[0])}.json').read_text(encoding='utf-8'))
    top = pd.Series(info['gain']).sort_values(ascending=False)
    added = set(STAGE1[cond]) - set(BASE)
    print(f"--- {cond}: " + ', '.join(f"{'*' if f in added else ''}{f} {v / top.sum():.0%}" for f, v in top.head(8).items()))
print('(* = 이 조건에서 얹은 피처)')

### 1단계 읽는 법
- **판정 표의 ↑** 가 붙은 묶음이 후보. `패턴P@0.1%`(같은 알림 수에서 패턴 precision) 와 `무패턴R@1%` 를 먼저 본다. 패턴 recall 은 이미 높아서 차이가 잘 안 난다
- **`비ACH알림@0.1%`**: 결제방식을 뺐으니 `base` 에서 비ACH 오탐이 많이 나올 수 있다 (07 의 `no_pf`). 어떤 묶음을 얹었을 때 이 숫자가 크게 줄면, 그 묶음이 결제방식이 하던 일(비ACH 거래 배제) 을 대신하는 것이다
- 표준편차가 0 에 가까우면 시드가 결과를 거의 안 흔든다는 뜻 (학습 표본이 고정이라 시드는 bagging 만 바꾼다). 그때는 차이의 크기 자체를 보고 판단한다
- 효과가 비슷하면 열 수가 적은 묶음을 고른다

## 6. 2단계 — 살아남은 묶음을 합친 뒤 하나씩 빼기
1단계에서 고른 묶음 글자를 `KEEP` 에 적는다. 따로 얹을 땐 올랐지만 같이 넣으면 겹치는 묶음을 여기서 거른다.
`keep` = 공통 + KEEP 전부, `keep-X` = 거기서 X 묶음만 뺀 것. `keep` 보다 `keep-X` 가 **내려가야** X 가 필요한 묶음이다

In [ ]:
KEEP = []                      # 예: ['A', 'C', 'F'] — 1단계 판정을 보고 채운다
STAGE2 = {}
if KEEP:
    keep_feats = BASE + [c for k in KEEP for c in GROUPS[k]]
    STAGE2 = {'keep': keep_feats, **{f'keep-{k}': [c for c in keep_feats if c not in GROUPS[k]] for k in KEEP}}
{k: len(v) for k, v in STAGE2.items()} or 'KEEP 을 채우고 다시 실행'

In [ ]:
if STAGE2:
    train_runs(STAGE2); score_runs(STAGE2)
    runs2 = eval_runs(STAGE2)
    runs2.to_csv(OUT_DIR / 'stage2_runs.csv', index=False, encoding='utf-8-sig')
    mean2, std2, delta2, verdict2 = summarize(runs2, 'keep')
    mean2.to_csv(OUT_DIR / 'stage2_mean.csv', encoding='utf-8-sig'); std2.to_csv(OUT_DIR / 'stage2_std.csv', encoding='utf-8-sig')
    judge2 = pd.concat({'차이': delta2.round(4), '판정': verdict2}, axis=1); judge2.to_csv(OUT_DIR / 'stage2_judge.csv', encoding='utf-8-sig')
    display(mean2.round(4)); display(std2.round(4)); display(judge2)

### 2단계 읽는 법 → 다음
- `keep-X` 의 판정이 **↓** 면 X 는 남긴다. **·** 면 X 는 다른 묶음과 겹치므로 빼도 된다 (열 수가 줄어든다)
- 최종 피처 셋 = 공통 17종 + 남긴 묶음. 이 셋으로 ① 결제방식 있음 / 없음 두 수치를 같이 내고 ② 태훈님 HGB · GNN 에서 한 번 확인한다
- 이 노트북의 v10 피처는 구간 끝 기준(미래 포함) 이다. 남긴 묶음은 내 규칙(전날까지) 으로 다시 계산해 `features_bc` 에 넣고 07 방식(학습 전체) 으로 한 번 더 확인한다